# 04 — Generator Fine-Tuning (DoRA)

This notebook fine-tunes the **generator** model using **DoRA adapters** (via PEFT) on `data/processed/generator_train.jsonl`.

**Input schema (JSONL)**: each row must contain: `system`, `user`, `context`, `assistant`.

**Output**: adapter + tokenizer saved to `models/generator_dora/`.


In [ ]:
import os
import sys
import json
from pathlib import Path

print('Python:', sys.version)
print('CWD:', os.getcwd())

ROOT = Path('.').resolve()
print('ROOT:', ROOT)

train_path = ROOT / 'data' / 'processed' / 'generator_train.jsonl'
print('train_path exists:', train_path.exists(), str(train_path))
assert train_path.exists(), 'Missing data/processed/generator_train.jsonl'

# Preview a few rows
with train_path.open('r', encoding='utf-8') as f:
    for i in range(3):
        line = f.readline().strip()
        obj = json.loads(line)
        print('--- row', i)
        print('keys:', list(obj.keys()))
        print('user:', obj.get('user','')[:120])
        print('context chars:', len(obj.get('context','')))
        print('assistant:', obj.get('assistant','')[:120])


## Train DoRA adapters

Recommended for an L4 (24GB): start with a small base model (default in script is `Qwen/Qwen2.5-1.5B-Instruct`).

If you want a larger model later, enable `--load_in_4bit` and consider reducing `--max_length`.


In [ ]:
# If running on Lightning AI, run from repo root.
# You can tweak args; start small.

!python -u src/generator/train_dora.py \
  --train data/processed/generator_train.jsonl \
  --out models/generator_dora \
  --epochs 1 \
  --batch_size 2 \
  --grad_accum 16 \
  --lr 2e-4 \
  --max_length 1024 \
  --bf16

In [ ]:
from pathlib import Path

out_dir = Path('models/generator_dora')
print('out_dir exists:', out_dir.exists())
if out_dir.exists():
    # show a few key artifacts
    for p in sorted(out_dir.glob('*'))[:50]:
        print(p.name)

# Minimal expectations: adapter + tokenizer files
expected_any = [
    'adapter_model.safetensors',
    'adapter_model.bin',
    'adapter_config.json',
    'tokenizer.json',
    'tokenizer_config.json',
]

found = {p.name for p in out_dir.glob('*')} if out_dir.exists() else set()
print('found expected:', {x: (x in found) for x in expected_any})


⏸️ TRAINING CHECKPOINT

Before proceeding to **DPO**, confirm that `models/generator_dora/` contains:
- `adapter_config.json`
- adapter weights (`adapter_model.safetensors` or `.bin`)
- tokenizer files (e.g., `tokenizer.json`)

Reply with the output of `ls -la models/generator_dora` from Lightning AI.